## Feature Extraction Notebook

### Imports

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold


from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

random_state = 42

In [ ]:
combined_raw = pd.read_csv("original_combined.csv") # load the merged raw dataset

### Create the 3 time windows: N400, P600, both

In [ ]:
electrodes = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6', 'T7', 'C3', 'Cz', 'C4', 'T8', 'TP9', 'CP5', 'CP1', 'CP2', 'CP6', 'TP10', 'P7', 'P3', 'Pz', 'P4', 'P8', 'PO9', 'O1', 'Oz', 'O2', 'PO10'] # common for all four original datasets

p600_df = combined_raw[
    (combined_raw["Timestamp"] >= 500) &
    (combined_raw["Timestamp"] <= 900)
] # select the P600 window from the merged dataset, [500, 900] ms

n400_df = combined_raw[
    (combined_raw["Timestamp"] >= 300) &
    (combined_raw["Timestamp"] <= 500)
] # select the N400 window from the merged dataset, [300, 500] ms

both_intervals_df = combined_raw[
    ((combined_raw["Timestamp"] >= 300) & (combined_raw["Timestamp"] <= 500)) |
    ((combined_raw["Timestamp"] >= 500) & (combined_raw["Timestamp"] <= 900))
] # select both the N400 and P600 intervals

### Extract features for the 3 datasets - comments in the first functions; all 3 do the same thing

In [ ]:
# helpers
def zero_crossing(x):
    x = np.asarray(x) # work with NumPy, feed timestamps in a fixed window as input
    return int(np.sum(np.diff(np.sign(x)) != 0))  # compute the sign of each value and if the sign changed between values, count it

# extract features for P600
def extract_features_p600(p600_df, electrodes):
    conditions = [
        (p600_df["Timestamp"] >= 500) & (p600_df["Timestamp"] < 600),
        (p600_df["Timestamp"] >= 600) & (p600_df["Timestamp"] < 700),
        (p600_df["Timestamp"] >= 700) & (p600_df["Timestamp"] < 800),
        (p600_df["Timestamp"] >= 800) & (p600_df["Timestamp"] <= 900),
    ] # create the four time windows from each features will be extracted

    bin_labels = [f"{s}_{e}" for s, e in [(500,600),(600,700),(700,800),(800,900)]] # labels

    p600_df = p600_df.copy() # copy to make sure nothing gets broken
    p600_df["bin"] = np.select(conditions, bin_labels, default="unknown") # select features corresponding to each bin

    p600_df["trial_uid"] = (
        p600_df["site"] + "__" +
        p600_df["subject_global"] + "__" +
        p600_df["Trial"].astype(str)
    ) # create unique identifiers: originalDataset__subjectID__associatedTrial

    group_keys = ["trial_uid", "site", "subject_global", "Trial", "bin"]
    grouped = p600_df.groupby(group_keys, sort=False) # group by relevant featurres

    # compute mean and std per electrode per group, then flatten multi-index columns to "electrode_mean/std"
    agg_df = grouped[electrodes].agg(["mean", "std"])
    agg_df.columns = ["_".join(col) for col in agg_df.columns]
    agg_df = agg_df.reset_index()

    def zc_agg(x):
        return zero_crossing(x.values)

    # compute zerro-crossings per electrode per group and aggregate
    zc_df = grouped[electrodes].agg(zc_agg)
    zc_df.columns = [f"{c}_zc" for c in zc_df.columns]
    zc_df = zc_df.reset_index()

    # reorder by cloze in each trial
    cloze_df = grouped["Cloze"].first().reset_index()

    # Merge mean, std, zc, and cloze
    features_long = agg_df.merge(zc_df, on=group_keys).merge(cloze_df, on=group_keys)

    id_cols = ["trial_uid", "site", "subject_global", "Trial", "Cloze"]
    feature_cols = [c for c in features_long.columns if c not in id_cols + ["bin"]]
    
    # prelucrations on the data for it to be in a proper format
    features_wide = features_long.pivot_table(
        index=id_cols,
        columns="bin",
        values=feature_cols,
        aggfunc="first"
    )

    features_wide.columns = [
        f"{feat}_{bin_range}"
        for feat, bin_range in features_wide.columns
    ]

    features_wide = features_wide.reset_index()

    # only necessary for building the features
    features_wide = features_wide.drop(columns=["trial_uid"])

    # sort for consistency
    features_wide = features_wide.sort_values(
        ["site", "subject_global", "Trial"]).reset_index(drop=True)

    # ensure that the trial index does not reset with each new dataset
    features_wide["trial_global"] = features_wide.index + 1 

    # reorderr columns for readability
    front_cols = ["trial_global", "site", "subject_global", "Trial", "Cloze"]
    rest_cols  = [c for c in features_wide.columns if c not in front_cols]
    features_wide = features_wide[front_cols + rest_cols]
    
    # return p600 features
    return features_wide

features_df_600 = extract_features_p600(p600_df, electrodes)
print(features_df_600.shape)
print(features_df_600.head()) # for inspection
features_df_600.to_csv("p600_features.csv", index=False) # save


def extract_features_n400(n400_df, electrodes):
    conditions = [
        (n400_df["Timestamp"] >= 300) & (n400_df["Timestamp"] < 400),
        (n400_df["Timestamp"] >= 400) & (n400_df["Timestamp"] <= 500),
    ]

    bin_labels = [f"{s}_{e}" for s, e in [(300,400),(400,500)]]
    n400_df = n400_df.copy()
    n400_df["bin"] = np.select(conditions, bin_labels, default="unknown")

    n400_df["trial_uid"] = (
        n400_df["site"] + "__" +
        n400_df["subject_global"] + "__" +
        n400_df["Trial"].astype(str)
    )

    group_keys = ["trial_uid", "site", "subject_global", "Trial", "bin"]

    grouped = n400_df.groupby(group_keys, sort=False)

    agg_df = grouped[electrodes].agg(["mean", "std"])
    agg_df.columns = ["_".join(col) for col in agg_df.columns]  # e
    agg_df = agg_df.reset_index()

    def zc_agg(x):
        return zero_crossing(x.values)

    zc_df = grouped[electrodes].agg(zc_agg)
    zc_df.columns = [f"{c}_zc" for c in zc_df.columns]
    zc_df = zc_df.reset_index()

    cloze_df = grouped["Cloze"].first().reset_index()

    # Merge mean, std, zc, and cloze
    features_long = agg_df.merge(zc_df, on=group_keys).merge(cloze_df, on=group_keys)

    id_cols = ["trial_uid", "site", "subject_global", "Trial", "Cloze"]
    feature_cols = [c for c in features_long.columns if c not in id_cols + ["bin"]]

    features_wide = features_long.pivot_table(
        index=id_cols,
        columns="bin",
        values=feature_cols,
        aggfunc="first"
    )

    features_wide.columns = [
        f"{feat}_{bin_range}"
        for feat, bin_range in features_wide.columns
    ]

    features_wide = features_wide.reset_index()

    features_wide = features_wide.drop(columns=["trial_uid"])

    features_wide = features_wide.sort_values(
        ["site", "subject_global", "Trial"]).reset_index(drop=True)

    features_wide["trial_global"] = features_wide.index + 1

    front_cols = ["trial_global", "site", "subject_global", "Trial", "Cloze"]
    rest_cols  = [c for c in features_wide.columns if c not in front_cols]
    features_wide = features_wide[front_cols + rest_cols]

    return features_wide

features_df_400 = extract_features_n400(n400_df, electrodes)
print(features_df_400.shape)
print(features_df_400.head())
features_df_400.to_csv("n400_features.csv", index=False) # save


def extract_features_both(both_df, electrodes):

    conditions = [
        (both_df["Timestamp"] >= 300) & (both_df["Timestamp"] < 400),
        (both_df["Timestamp"] >= 400) & (both_df["Timestamp"] <= 500),
        (both_df["Timestamp"] >= 500) & (both_df["Timestamp"] < 600),
        (both_df["Timestamp"] >= 600) & (both_df["Timestamp"] < 700),
        (both_df["Timestamp"] >= 700) & (both_df["Timestamp"] < 800),
        (both_df["Timestamp"] >= 800) & (both_df["Timestamp"] <= 900),
    ]

    bin_labels = [f"{s}_{e}" for s, e in [(300, 400), (400, 500),(500, 600), (600,700),(700,800),(800,900)]]

    both_df = both_df.copy()
    both_df["bin"] = np.select(conditions, bin_labels, default="unknown")

    both_df["trial_uid"] = (
        both_df["site"] + "__" +
        both_df["subject_global"] + "__" +
        both_df["Trial"].astype(str)
    )

    group_keys = ["trial_uid", "site", "subject_global", "Trial", "bin"]

    grouped = both_df.groupby(group_keys, sort=False)

    agg_df = grouped[electrodes].agg(["mean", "std"])
    agg_df.columns = ["_".join(col) for col in agg_df.columns] 
    agg_df = agg_df.reset_index()

    def zc_agg(x):
        return zero_crossing(x.values)

    zc_df = grouped[electrodes].agg(zc_agg)
    zc_df.columns = [f"{c}_zc" for c in zc_df.columns]
    zc_df = zc_df.reset_index()

    cloze_df = grouped["Cloze"].first().reset_index()

    features_long = agg_df.merge(zc_df, on=group_keys).merge(cloze_df, on=group_keys)

    id_cols = ["trial_uid", "site", "subject_global", "Trial", "Cloze"]
    feature_cols = [c for c in features_long.columns if c not in id_cols + ["bin"]]

    features_wide = features_long.pivot_table(
        index=id_cols,
        columns="bin",
        values=feature_cols,
        aggfunc="first"
    )

    features_wide.columns = [
        f"{feat}_{bin_range}"
        for feat, bin_range in features_wide.columns
    ]
    features_wide = features_wide.reset_index()

    features_wide = features_wide.drop(columns=["trial_uid"])

    features_wide = features_wide.sort_values(
        ["site", "subject_global", "Trial"]).reset_index(drop=True)

    features_wide["trial_global"] = features_wide.index + 1

    front_cols = ["trial_global", "site", "subject_global", "Trial", "Cloze"]
    rest_cols  = [c for c in features_wide.columns if c not in front_cols]
    features_wide = features_wide[front_cols + rest_cols]

    return features_wide

features_df_both = extract_features_both(both_intervals_df, electrodes)
print(features_df_both.shape)
print(features_df_both.head())
features_df_both.to_csv("both_intervals_features.csv", index=False) # save

(9463, 389)
   trial_global  site subject_global  Trial  Cloze  C3_mean_500_600  \
0             1  adbc         adbc_1      1    1.0        21.790390   
1             2  adbc         adbc_1      2    0.8        -0.826331   
2             3  adbc         adbc_1      3    0.6        -4.368479   
3             4  adbc         adbc_1      4    0.8        -7.889166   
4             5  adbc         adbc_1      5    0.8        30.626768   

   C3_mean_600_700  C3_mean_700_800  C3_mean_800_900  C3_std_500_600  ...  \
0        17.341469        18.204878        17.366291        2.736003  ...   
1        -4.269750         0.302962       -11.611648        3.144784  ...   
2        -4.452870         6.647142         7.861491        3.335385  ...   
3       -11.306078        -9.897429       -10.055741        4.621188  ...   
4        22.028609        30.505176        20.122523       11.439358  ...   

   TP9_mean_700_800  TP9_mean_800_900  TP9_std_500_600  TP9_std_600_700  \
0        -15.199311    

In [5]:
p600_f = pd.read_csv("p600_features.csv")
p600_f.head()

,trial_global,site,subject_global,Trial,Cloze,C3_mean_500_600,C3_mean_600_700,C3_mean_700_800,C3_mean_800_900,C3_std_500_600,...,TP9_mean_700_800,TP9_mean_800_900,TP9_std_500_600,TP9_std_600_700,TP9_std_700_800,TP9_std_800_900,TP9_zc_500_600,TP9_zc_600_700,TP9_zc_700_800,TP9_zc_800_900
0,1,adbc,adbc_1,1,1.0,21.790390,17.341469,18.204878,17.366291,2.736003,...,-15.199311,-15.437490,3.345646,4.945860,3.360896,4.824074,0,0,0,0
1,2,adbc,adbc_1,2,0.8,-0.826331,-4.269750,0.302962,-11.611648,3.144784,...,-2.387513,8.658370,6.461336,6.627418,7.245864,6.786317,1,2,2,1
2,3,adbc,adbc_1,3,0.6,-4.368479,-4.452870,6.647142,7.861491,3.335385,...,-19.863894,-18.533479,5.014977,8.504542,8.919834,5.704500,2,1,0,0
3,4,adbc,adbc_1,4,0.8,-7.889166,-11.306078,-9.897429,-10.055741,4.621188,...,-17.482162,-17.125801,8.223771,11.462112,9.280120,3.914210,3,0,0,0
4,5,adbc,adbc_1,5,0.8,30.626768,22.028610,30.505176,20.122523,11.439358,...,-19.105828,-3.888493,8.829419,4.842220,2.934228,3.320938,0,0,0,4


In [20]:
n400_f = pd.read_csv("n400_features.csv")
n400_f.head()

,trial_global,site,subject_global,Trial,Cloze,C3_mean_300_400,C3_mean_400_500,C3_std_300_400,C3_std_400_500,C3_zc_300_400,...,TP10_std_300_400,TP10_std_400_500,TP10_zc_300_400,TP10_zc_400_500,TP9_mean_300_400,TP9_mean_400_500,TP9_std_300_400,TP9_std_400_500,TP9_zc_300_400,TP9_zc_400_500
0,1,adbc,adbc_1,1,1.0,16.256696,16.801147,4.080551,6.078874,0,...,5.519957,9.472803,0,4,-19.393025,-8.468638,7.513565,8.741156,0,2
1,2,adbc,adbc_1,2,0.8,-1.234983,-4.391995,4.186938,7.225523,1,...,6.686204,12.218055,1,1,-2.919245,2.482437,7.617191,12.491124,1,1
2,3,adbc,adbc_1,3,0.6,2.154637,-1.554956,5.403830,4.685967,1,...,8.162028,7.373363,1,3,-9.255632,-1.807674,8.640326,7.796310,2,2
3,4,adbc,adbc_1,4,0.8,-4.509541,-1.406189,9.674068,5.923248,5,...,11.210492,7.164904,2,0,-16.219123,-15.686837,9.365731,6.416100,0,1
4,5,adbc,adbc_1,5,0.8,16.712741,22.776078,12.525536,10.865521,2,...,12.292787,10.099486,3,0,-17.358283,-14.823142,12.655939,16.887707,1,1


In [6]:
both_f = pd.read_csv("both_intervals_features.csv")
both_f.head()

,trial_global,site,subject_global,Trial,Cloze,C3_mean_300_400,C3_mean_400_500,C3_mean_500_600,C3_mean_600_700,C3_mean_700_800,...,TP9_std_500_600,TP9_std_600_700,TP9_std_700_800,TP9_std_800_900,TP9_zc_300_400,TP9_zc_400_500,TP9_zc_500_600,TP9_zc_600_700,TP9_zc_700_800,TP9_zc_800_900
0,1,adbc,adbc_1,1,1.0,16.256696,16.801147,21.769633,17.341469,18.204878,...,3.290613,4.945860,3.360896,4.824074,0,2,0,0,0,0
1,2,adbc,adbc_1,2,0.8,-1.234983,-4.391995,-0.769779,-4.269750,0.302962,...,6.241388,6.627418,7.245864,6.786317,1,1,1,2,2,1
2,3,adbc,adbc_1,3,0.6,2.154637,-1.554956,-4.332789,-4.452870,6.647142,...,5.040498,8.504542,8.919834,5.704500,2,2,2,1,0,0
3,4,adbc,adbc_1,4,0.8,-4.509541,-1.406189,-7.817966,-11.306078,-9.897429,...,8.212236,11.462112,9.280120,3.914210,0,1,3,0,0,0
4,5,adbc,adbc_1,5,0.8,16.712741,22.776078,30.727047,22.028610,30.505176,...,8.858786,4.842220,2.934228,3.320938,1,1,0,0,0,4
